# Enterprise Software User Complaints Generator

## Project Overview
This notebook is designed to generate a synthetic dataset of **10 user complaints** regarding enterprise software systems.

**The workflow includes:**
1.  **System & Industry Generation:** Leveraging OpenAI to create realistic lists of enterprise software types and industries.
2.  **Workflow Simulation:** Generating detailed technical workflows and architectures.
3.  **Malfunction Injection:** Randomly selecting components to simulate malfunctions with varying severity levels.
4.  **User Complaint Synthesis:** Creating natural language user complaints based on the generated malfunctions and specific user personas.

## Step 1: Initialize Environment

### Subtask: Load OpenAI API Key
We need to securely load the OpenAI API key to authenticate our requests. This will be retrieved from the notebook's secret storage or environment variables.

> **Reasoning:**
> Hard-coding API keys is a security risk. By importing the `os` module and retrieving the key from environment variables, we ensure secure access to the OpenAI API without exposing sensitive credentials in the code.

In [ ]:
import os
from openai import OpenAI

In [41]:
# Initialize the OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Step 2: Generate Enterprise Software Systems

### Subtask: Create Systems List
Use OpenAI to generate a list of 10 enterprise software systems (e.g., ERP, CRM). The output will be a structured JSON array.

> **Reasoning:**
> We need a diverse base of software systems to create realistic scenarios. We define a strict JSON schema in the prompt to ensure the output is machine-readable and ready for the next steps of the pipeline.


In [ ]:
import json

In [ ]:


# Create a prompt for generating enterprise software systems
prompt_text = """Generate a list of 10 types of enterprise software systems as a simple JSON array. Each element should have a 'system' name and a 'description'.
Top level element in JSON output should be 'systems'
Example format:
[
  {
    "system": "ERP System",
    "description": "A comprehensive suite for managing core business processes."
  },
  {
    "system": "CRM",
    "description": "A customeer relation managmenet system"
  }
]
"""

# Call the OpenAI Chat Completions API
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant designed to output JSON."
        },
        {
            "role": "user",
            "content": prompt_text,
        }
    ],
    model="gpt-3.5-turbo-1106", # Using a model known to be good at JSON output
    response_format={ "type": "json_object" }
)

# Parse the JSON response
response_content = chat_completion.choices[0].message.content
enterprise_systems = json.loads(response_content)

print("Generated enterprise systems from OpenAI API.",enterprise_systems)

Generated enterprise systems from OpenAI API. {'systems': [{'system': 'ERP System', 'description': 'A comprehensive suite for managing core business processes.'}, {'system': 'CRM', 'description': 'A customer relationship management system for managing interactions with current and potential customers.'}, {'system': 'HRM System', 'description': 'A system for managing human resources, including payroll, benefits, and employee performance.'}, {'system': 'Supply Chain Management System', 'description': 'A system for overseeing the flow of goods, information, and finances as they move from supplier to manufacturer to wholesaler to retailer to customer.'}, {'system': 'Business Intelligence System', 'description': 'A system for analyzing and reporting on business data to help make informed decisions.'}, {'system': 'Enterprise Content Management System', 'description': "A system for organizing and managing an organization's documents, and other content that relates to the organization's proces

**Reasoning**:
The previous step successfully generated a list of enterprise software systems and stored it in the `enterprise_systems` variable. The next instruction is to save this data as a JSON file to disk.



In [ ]:
import json

In [ ]:
# Save the generated list of enterprise software systems as a JSON file
with open('enterprise_systems.json', 'w') as f:
    json.dump(enterprise_systems, f, indent=2)

print("Enterprise systems saved to 'enterprise_systems.json'.")

Enterprise systems saved to 'enterprise_systems.json'.


## Step 3: Generate Industries

### Subtask: Create Industries List
Similar to the systems step, we will use OpenAI to generate a list of 10 industries (e.g., Healthcare, Finance) to provide context for the software usage.

> **Reasoning:**
> Mixing different industries with different software systems allows us to create specific, context-aware workflow descriptions later (e.g., "ERP in Healthcare" vs "ERP in Retail").

In [ ]:
import json

In [ ]:
# Create a prompt for generating industries
prompt_text_industries = """Generate a list of 10 industries. Each industry should have an 'industry' name and a 'description'. The output should be a JSON object containing a 'industries' key, which is a JSON array of objects, where each object has 'industry' and 'description' as keys.
top level element in josn should be industries
Example format:
{
  "industries": [
    {
      "industry": "Financial Services",
      "description": "Sector dealing with money management, including banks, credit unions, and investment firms."
    }
  ]
}
"""

# Call the OpenAI Chat Completions API
chat_completion_industries = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant designed to output JSON."
        },
        {
            "role": "user",
            "content": prompt_text_industries,
        }
    ],
    model="gpt-3.5-turbo-1106", # Using a model known to be good at JSON output
    response_format={ "type": "json_object" }
)

# Parse the JSON response
response_content_industries = chat_completion_industries.choices[0].message.content
industries = json.loads(response_content_industries)

print("Generated industries from OpenAI API.",industries)

Generated industries from OpenAI API. {'industries': [{'industry': 'Information Technology', 'description': 'Industry dealing with the technology, development, and management of computer systems, software, and networks.'}, {'industry': 'Healthcare', 'description': 'Sector focused on the maintenance or improvement of health via the diagnosis, treatment, and prevention of disease, illness, injury, and other physical and mental impairments in humans.'}, {'industry': 'Automotive', 'description': 'Industry involved in the design, development, manufacturing, marketing, and selling of motor vehicles.'}, {'industry': 'Retail', 'description': 'Sector that encompasses the process of selling consumer goods or services to customers through multiple channels of distribution.'}, {'industry': 'Hospitality', 'description': 'Industry encompassing lodging, event planning, theme parks, transportation, cruise line, and additional fields within the tourism industry.'}, {'industry': 'Energy', 'description':

**Reasoning**:
The previous step successfully generated a list of industries and stored it in the `industries` variable. The next instruction is to save this data as a JSON file to disk.



In [ ]:
# Save the generated list of industries as a JSON file
with open('industries.json', 'w') as f:
    json.dump(industries, f, indent=2)

print("Industries saved to 'industries.json'.")

Industries saved to 'industries.json'.


## Step 4: Define Core Functions

### Function 1: `generate_workflow`
This function orchestrates the creation of a technical workflow description.

* **Inputs:** Lists of enterprise systems and industries.
* **Process:** Selects a random pair (System + Industry) and prompts the LLM to describe a technical workflow with specific components.
* **Output:** A dictionary containing the workflow name, description, and technical components.

> **Reasoning:**
> We encapsulate this logic in a function to make the code modular and reusable. The prompt is engineered to request specific component names and descriptions, which serve as the "targets" for our malfunctions.

In [ ]:
import random
from openai import OpenAI # client is already initialized, but for standalone clarity
import json

In [44]:
def generate_workflow(enterprise_systems: dict, industries: dict, client: OpenAI) -> tuple:
    """
    Generates a technical workflow description using OpenAI based on a random system and industry.

    This function selects a random enterprise system and industry, then constructs a prompt
    to generate a detailed technical architecture including specific components.

    Args:
        enterprise_systems (dict): Dictionary containing a list of system types.
        industries (dict): Dictionary containing a list of industries.
        client (OpenAI): Authenticated OpenAI client instance.

    Returns:
        tuple: A tuple containing:
            - workflow_description (dict): Parsed JSON with workflow name, description, and components.
            - selected_system (dict): The specific system chosen.
            - selected_industry (dict): The specific industry chosen.
    """
    # --- Step 1: Random Selection ---
    # We randomize the context to ensure a diverse dataset.
    selected_system = random.choice(enterprise_systems['systems'])
    selected_industry = random.choice(industries['industries'])

    system_name = selected_system['system']
    system_description = selected_system['description']
    industry_name = selected_industry['industry']
    industry_description = selected_industry['description']

    # --- Step 2: Prompt Engineering ---
    # We ask for a structured JSON output to ensure consistent parsing later.
    # The prompt explicitly requests 'workflow_components' which are crucial for the malfunction step.
    prompt_text = f"""Generate a concise workflow or architecture description with technical sub-systems and components for the enterprise software system '{system_name}' operating within the '{industry_name}' industry. 
    
    Context:
    - System Purpose: '{system_description}'
    - Industry Context: '{industry_description}'

    Requirements:
    - Output must be a valid JSON object.
    - Include a 'workflow_name', 'workflow_description'.
    - Include 'workflow_components': A JSON array of at least 5 key technical components (name and description).
    
    Example Structure:
    {{
      "workflow_name": "...",
      "workflow_description": "...",
      "workflow_components": [
        {{"name": "Component A", "description": "..."}},
        {{"name": "Component B", "description": "..."}}
      ]
    }}
    """

    # --- Step 3: API Call & Parsing ---
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a technical architect assistant designed to output strict JSON."
            },
            {
                "role": "user",
                "content": prompt_text,
            }
        ],
        model="gpt-3.5-turbo-1106", # Using a JSON-optimized model
        response_format={ "type": "json_object" }
    )

    # Parse the response string into a Python dictionary
    response_content = chat_completion.choices[0].message.content
    workflow_description = json.loads(response_content)

    print(f"Generated workflow for {system_name} in {industry_name}.")
    return workflow_description, selected_system, selected_industry

print("Defined the 'generate_workflow' function with improved documentation.")

Defined the 'generate_workflow' function with improved documentation.


### Function 2: `generate_malfunction`
This function simulates a software failure within the generated workflow.

* **Inputs:** A workflow description dictionary.
* **Process:** Randomly selects 1-3 components from the workflow and assigns a severity level (Critical, High, Medium).
* **Output:** A description of the malfunction and the specific affected component.

> **Reasoning:**
> By randomly selecting components from the previous step, we ensure the malfunction is technically consistent with the generated architecture.

In [ ]:
import random
from openai import OpenAI
import json

In [45]:
def generate_malfunction(workflow_description: dict, client: OpenAI) -> dict:
    """
    Simulates a software malfunction within a given workflow.

    It randomly selects a component from the workflow and a severity level,
    then asks the LLM to describe a realistic failure scenario.

    Args:
        workflow_description (dict): The workflow data containing 'workflow_components'.
        client (OpenAI): Authenticated OpenAI client instance.

    Returns:
        dict: A dictionary with the malfunction description, affected component details, and severity.
    """
    # --- Step 1: Component & Severity Selection ---
    # Extract the list of components generated in the previous step
    workflow_components = workflow_description['workflow_components']
    
    # Randomly select ONE component to be the source of failure
    selected_component = random.choice(workflow_components)

    # Randomly assign a severity level to vary the dataset
    severity = random.choice(['critical', 'high', 'medium'])

    # --- Step 2: Prompt Construction ---
    # The prompt injects the specific component and workflow context to make the error realistic.
    prompt_text = f"""Describe a malfunction or issue related to the following technical component:
    Component Name: {selected_component['name']}
    Component Description: {selected_component['description']}

    Context:
    Workflow: {workflow_description['workflow_name']}
    Description: {workflow_description['workflow_description']}

    Task:
    Generate a '{severity}' severity malfunction. Provide a realistic technical scenario.
    
    Output JSON format:
    {{
      "affected_component": {{ "name": "...", "description": "..." }},
      "malfunction_description": "Detailed technical description of the issue...",
      "severity": "{severity}"
    }}
    """

    # --- Step 3: API Call ---
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a QA engineer assistant designed to output strict JSON describing software faults."
            },
            {
                "role": "user",
                "content": prompt_text,
            }
        ],
        model="gpt-3.5-turbo-1106",
        response_format={ "type": "json_object" }
    )

    response_content = chat_completion.choices[0].message.content
    malfunction_details = json.loads(response_content)

    print(f"Generated {severity} malfunction for: {selected_component['name']}")
    return malfunction_details

print("Defined the 'generate_malfunction' function with improved documentation.")

Defined the 'generate_malfunction' function with improved documentation.


### Function 3: `generate_user_description`
This function translates the technical malfunction into a user-facing complaint.

* **Inputs:** Malfunction details.
* **Process:** Assigns user personas (experience level, communication style) and generates a natural language complaint.
* **Output:** A user complaint string and the attributes of the user.

> **Reasoning:**
> This step bridges the gap between backend errors and user experience. It simulates how different users might describe the same technical problem differently.

In [ ]:
import random
from openai import OpenAI
import json

In [46]:
def generate_user_description(malfunction_details: dict, client: OpenAI) -> dict:
    """
    Translates a technical malfunction into a natural language user complaint.

    This function assigns random user attributes (experience, style) and generates
    a complaint text that reflects how a user would perceive the underlying technical fault.

    Args:
        malfunction_details (dict): The technical details of the failure.
        client (OpenAI): Authenticated OpenAI client.

    Returns:
        dict: A dictionary containing the 'user_complaint' text and 'user_attributes'.
    """
    # --- Step 1: User Persona Generation ---
    # We mix and match attributes to create diverse user voices (e.g., Angry Expert vs. Confused Novice)
    user_experience = random.choice(['novice', 'experienced', 'expert'])
    communication_style = random.choice(['apologetic', 'calm', 'frustrated', 'urgent'])
    issue_category = random.choice(['cannot complete task', 'slowdown', 'data inconsistency', 'security concern'])

    # Prepare data for prompt
    malfunction_desc = malfunction_details['malfunction_description']
    severity = malfunction_details['severity']
    comp_data = malfunction_details['affected_component']
    affected_component_str = f"{comp_data['name']} ({comp_data['description']})"

    # --- Step 2: Prompt Engineering ---
    # Constraint: The user usually doesn't know the exact component name, so we ask the LLM to avoid it in the complaint.
    prompt_text = f"""Generate a user-friendly complaint description based on this technical malfunction.
    
    Technical Context:
    - Issue: {malfunction_desc}
    - Severity: {severity}
    - Underlying Component: {affected_component_str}

    User Persona:
    - Experience Level: {user_experience}
    - Tone/Style: {communication_style}
    - Perceived Impact: {issue_category}

    Instructions:
    - The complaint should sound realistic for this persona.
    - DO NOT use the exact technical name of the component (users rarely know this).
    - Focus on the symptom and the impact on their work.

    Output JSON format:
    {{
      "user_complaint": "I can't believe...",
      "user_attributes": {{
        "user_experience": "{user_experience}",
        "communication_style": "{communication_style}",
        "issue_category": "{issue_category}"
      }}
    }}
    """

    # --- Step 3: API Call ---
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a creative writer designed to output JSON representing user feedback."
            },
            {
                "role": "user",
                "content": prompt_text,
            }
        ],
        model="gpt-3.5-turbo-1106",
        response_format={ "type": "json_object" }
    )

    response_content = chat_completion.choices[0].message.content
    user_complaint_details = json.loads(response_content)

    print(f"Generated complaint for {user_experience} user ({communication_style}).")
    return user_complaint_details

print("Defined the 'generate_user_description' function with improved documentation.")

Defined the 'generate_user_description' function with improved documentation.


## Step 5: Execute Generation Pipeline

### Subtask: Generate Dataset
We will now run the pipeline to generate 10 complete examples. Each iteration calls the three functions defined above in sequence.

> **Reasoning:**
> Aggregating the outputs of all functions into a single list of dictionaries allows us to create a structured dataset that links the system, industry, technical fault, and user complaint together.

In [ ]:
# List to store the final dataset
user_complaints_examples = []
NUM_EXAMPLES = 10 

print(f"Starting generation of {NUM_EXAMPLES} examples...")

for i in range(NUM_EXAMPLES):
    print(f"\n--- Generating example {i+1}/{NUM_EXAMPLES} ---")
    
    # 1. Generate the infrastructure context
    workflow_desc, sys_info, ind_info = generate_workflow(enterprise_systems, industries, client)

    # 2. Inject a fault into that infrastructure
    malfunction_info = generate_malfunction(workflow_desc, client)

    # 3. Simulate the user's reaction to that fault
    complaint_info = generate_user_description(malfunction_info, client)

    # 4. Data Aggregation
    # Combine all layers of data into a single structured record
    full_example = {
        "example_id": i + 1,
        "system_context": sys_info,
        "industry_context": ind_info,
        "workflow_context": workflow_desc,
        "technical_fault": malfunction_info,
        "user_feedback": complaint_info
    }

    user_complaints_examples.append(full_example)

# Preview the result
print("\n--- Generation Complete ---")
print(f"Total examples generated: {len(user_complaints_examples)}")
# print(json.dumps(user_complaints_examples[0], indent=2)) # Uncomment to see the first exampl

## Step 6: Data Export

### Subtask: Save to JSON
Finally, we save the generated dataset to a local file (`generated_dataset.json`).

> **Reasoning:**
> Persisting the data allows for offline analysis, model training, or sharing the dataset without needing to regenerate it (and pay for API calls) every time.

In [ ]:
import json

output_filename = "generated_dataset.json"

with open(output_filename, 'w') as f:
    json.dump(user_complaints_examples, f, indent=4)

print(f"Successfully saved data to {output_filename}")

Successfully saved data to generated_dataset.json


# Project Summary & Key Takeaways

## 1. Executive Summary
This notebook successfully orchestrated an end-to-end pipeline to generate a synthetic dataset of **10 enterprise software user complaints**. By leveraging OpenAI's LLM, we simulated complex technical environments, injected specific malfunctions, and synthesized realistic user feedback.

## 2. Key Outputs
The following artifacts were generated and saved locally:
* `enterprise_systems.json`: A catalog of enterprise software types.
* `industries.json`: A list of diverse industry contexts.
* `generated_dataset.json`: The final dataset containing structured examples linking technical faults to user complaints.

## 3. Technical Highlights
* **Modular Design:** The process was broken down into distinct, reusable functions (`generate_workflow`, `generate_malfunction`, `generate_user_description`).
* **Structured Data:** Enforced JSON formatting in LLM responses ensured high data integrity and ease of parsing.
* **Contextual Depth:** Each complaint is grounded in a specific technical architecture and industry context, providing rich metadata for analysis.

## 4. Next Steps
* **Data Analysis:** Load `generated_dataset.json` into a Pandas DataFrame to analyze the distribution of severity levels or issue categories.
* **Model Training:** Use this synthetic dataset to fine-tune a classifier for automated ticket routing.
* **Scale Up:** Increase the loop range to generate a larger dataset (e.g., 100+ examples) for more robust testing.